In [ ]:
display(
    spark.read.json("Files/bronze/products").limit(5)
)

In [ ]:
products_df = spark.read.json("Files/bronze/products")

display(products_df)

In [ ]:
products_df.printSchema()

In [ ]:
from pyspark.sql.functions import explode

products_flat_df = products_df.select(
    explode("products").alias("product")
)

display(products_flat_df.limit(5))

In [ ]:
products_flat_df = products_flat_df.select(
    "product.*"
)

display(products_flat_df.limit(5))

In [ ]:
from pyspark.sql.functions import col, sum

products_flat_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in products_flat_df.columns
    ]
).show()

In [ ]:
products_flat_df.select("id").distinct().count()

In [ ]:
silver_products_df = products_flat_df.select(
    "id",
    "title",
    "category",
    "brand",
    "sku",
    "price",
    "discountPercentage",
    "rating",
    "stock",
    "availabilityStatus",
    "minimumOrderQuantity",
    "returnPolicy",
    "warrantyInformation",
    "shippingInformation"
)

display(silver_products_df.limit(5))

In [ ]:
from pyspark.sql.functions import current_timestamp

silver_products_df = silver_products_df.withColumn(
    "silver_processed_at",
    current_timestamp()
)

display(silver_products_df.limit(5))

In [ ]:
silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_products")

In [ ]:
users_df = spark.read.json("Files/bronze/users")

display(users_df.limit(5))

In [ ]:
from pyspark.sql.functions import explode

users_flat_df = users_df.select(
    explode("users").alias("user")
)

display(users_flat_df.limit(5))

In [ ]:
users_flat_df = users_flat_df.select(
    "user.*"
)

display(users_flat_df.limit(5))

In [ ]:
silver_users_df = users_flat_df.select(
    "id",
    "firstName",
    "lastName",
    "age",
    "gender",
    "email",
    "phone",
    "username",
    "birthDate",
    "bloodGroup",
    "eyeColor",
    "height",
    "weight",
    "address",
    "company"
)

display(silver_users_df.limit(5))

In [ ]:
silver_users_df.count()

In [ ]:
silver_users_df.select("id").distinct().count()

In [ ]:
from pyspark.sql.functions import current_timestamp

silver_users_df = silver_users_df.withColumn(
    "silver_processed_at",
    current_timestamp()
)

display(silver_users_df.limit(5))

In [ ]:
silver_users_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_users")

In [ ]:
carts_df = spark.read.json("Files/bronze/carts")

display(carts_df.limit(5))

In [ ]:
from pyspark.sql.functions import explode

carts_flat_df = carts_df.select(
    explode("carts").alias("cart")
)

display(carts_flat_df.limit(5))

In [ ]:
carts_flat_df = carts_flat_df.select(
    "cart.*"
)

display(carts_flat_df.limit(5))

In [ ]:
cart_items_df = carts_flat_df.select(
    "id",
    "userId",
    explode("products").alias("product"),
    "total",
    "discountedTotal",
    "totalProducts",
    "totalQuantity"
)

display(cart_items_df.limit(5))

In [ ]:
cart_items_df = cart_items_df.select(
    "id",
    "userId",
    "product.*",
    "total",
    "discountedTotal",
    "totalProducts",
    "totalQuantity"
)

display(cart_items_df.limit(5))

In [ ]:
from pyspark.sql.functions import explode

cart_items_exploded_df = carts_flat_df.select(
    "id",
    "userId",
    explode("products").alias("product")
)

display(cart_items_exploded_df.limit(5))

In [ ]:
from pyspark.sql.functions import col

cart_items_df = cart_items_exploded_df.select(
    col("id").alias("cart_id"),
    col("userId").alias("user_id"),
    col("product.id").alias("product_id"),
    col("product.quantity").alias("quantity"),
    col("product.price").alias("price"),
    col("product.discountPercentage").alias("discount_percentage"),
    col("product.discountedTotal").alias("discounted_total")
)

display(cart_items_df.limit(5))

In [ ]:
cart_items_df.count()

In [ ]:
from pyspark.sql.functions import col, sum

cart_items_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in cart_items_df.columns
    ]
).show()

In [ ]:
from pyspark.sql.functions import col

cart_items_df = cart_items_df.withColumn(
    "gross_amount",
    col("price") * col("quantity")
)

display(cart_items_df.limit(5))

In [ ]:
cart_items_df = cart_items_df.withColumn(
    "discount_amount",
    col("gross_amount") - col("discounted_total")
)

display(cart_items_df.limit(5))

In [ ]:
from pyspark.sql.functions import current_timestamp

cart_items_df = cart_items_df.withColumn(
    "silver_processed_at",
    current_timestamp()
)

display(cart_items_df.limit(5))

In [ ]:
cart_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_cart_items")

In [ ]:
silver_products = spark.table("silver_products")

display(silver_products.limit(5))